In [ ]:
import pandas as pd
import seaborn as sns

from plotly import express as px
from plotly.figure_factory import create_annotated_heatmap
from plotly import graph_objects as go

from sklearn.preprocessing import LabelEncoder
from matplotlib import pyplot as plt

In [ ]:
data = pd.read_csv('data/heart_2020_cleaned.csv')
data.head()

In [ ]:
d = ['AgeCategory', 'Race', 'Diabetic', 'GenHealth']
c = data.select_dtypes('object').drop(d, axis=1).columns
for col in c:
    data[f'{col}_no'] = data[col].replace(['No', 'Female'], 0).replace(['Yes', 'Male'], 1)
for col in d:
    data[f'{col}_no'] = LabelEncoder().fit_transform(data[f'{col}'])

In [ ]:
plt.figure(figsize=(10,10))
grid = sns.FacetGrid(data=data,col='HeartDisease', row='Sex', aspect=1)
fig = grid.map(sns.countplot, 'DiffWalking', palette='Set1', order=data.Stroke.unique())
fig.add_legend()
plt.show()

# Numerous features

In [ ]:
px.box(data_frame=data, x='AgeCategory', y='BMI', color='HeartDisease')

+ Bất chấp độ tuổi, tỉ lệ bị bệnh tim vẫn tập trung ở BMI trên 20 dưới 30 (bình thường)
+ Người càng béo (BMI càng cao) vẫn bị bệnh tim khá nhiều và tập trung ở những người trung niên (>40 tuổi).

In [ ]:
fig = px.scatter(data, x="BMI", y="SleepTime",
                 color="HeartDisease",
                 log_x=True, size_max=60)
fig.show()

+ Tỉ lệ bị bệnh tim tập trung ở cả những người có chỉ số BMI bình thường (20-30) và thời gian ngủ bình thường.
+ Đối với những người ngủ nhiều (trên 10h) thì lại không phổ biến bệnh tim như người ngủ ít (3-5h).

In [ ]:
fig = px.scatter(data, x="PhysicalHealth", y="MentalHealth",
                 color="HeartDisease",
                 log_x=True, size_max=60)
fig.show()

+ Những ai bị bệnh về cơ thể càng lâu thì càng có nguy cơ bệnh tim.
+ Song, điều đó không đúng với bệnh tâm lý.

# Nomial features

In [ ]:
def plot_cate(col):
    plot = px.pie(data.replace({1:2, 0:1}), names=col, values=f'{col}_no', hole=0.6, title=f'Overall {col} in data',
                  color_discrete_sequence=px.colors.qualitative.T10)
    plot.update_traces(textfont=dict(color='#fff'))
    plot.update_layout(autosize=True, height=500, width=800,
                       margin=dict(t=80, b=30, l=70, r=40),
                       plot_bgcolor='#2d3035', paper_bgcolor='#2d3035',
                       title_font=dict(size=25, color='#a5a7ab', family="Muli, sans-serif"),
                       font=dict(color='#8a8d93'),
                       legend=dict(y=1.02, x=1)
                       )
    plot.show()

In [ ]:
for col in data.select_dtypes('object').columns:
    plot_cate(col)

+ Có thể thấy rõ rằng phần lớn các features đều mất cân bằng nghiêm trọng.
+ Nó sẽ gây khó khăn cho các phân tích về sau.

# Othes

In [ ]:
col_corr = [x for x in data.columns if x.endswith('_no')]
corr = data[col_corr].corr(method='pearson')
fig_heatmap = create_annotated_heatmap(
    z=corr.values,
    x=list(corr.columns),
    y=list(corr.index)
    annotation_text=corr.round(2).values,
    showscale=True)
fig_heatmap.update_layout(title= 'Correlation of whole Data',
                 plot_bgcolor='#2d3035', paper_bgcolor='#2d3035',
                        title_font=dict(size=25, color='#a5a7ab', family="Muli, sans-serif"),
                        font=dict(color='#8a8d93'))

Bảng tương quan của các features. Tuy nhiên đây chỉ là bản sơ bộ, cần tính toán kỹ lưỡng hơn.

In [ ]:
classname = px.histogram(data, x='AgeCategory', color='HeartDisease',
                         title='Heart Disease by Age Category', height=300,
                         category_orders={'Heart Disease': ['No', 'Yes']},
                         color_discrete_sequence=['#DB6574', '#03DAC5'],
                         barmode='group'
                         )
classname.update_yaxes(showgrid=False),
classname.update_xaxes(categoryorder='total descending')
classname.update_traces(hovertemplate=None)
classname.update_layout(margin=dict(t=100, b=0, l=70, r=40),
                        hovermode="x unified",
                    những người có kết quả sức khỏe tổng quát từ tốt trở lên đều có tỉ lệ lớn không bị bệnh tim    xaxis_tickangle=360,
                        xaxis_title=' ', yaxis_title=" ",
                        plot_bgcolor='#2d3035', paper_bgcolor='#2d3035',
                        title_font=dict(size=25, color='#a5a7ab', family="Muli, sans-serif"),
                        font=dict(color='#8a8d93'),
                        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
                          )

+ Ta có thể thấy rõ rằng người càng lớn tuổi càng có tỉ lệ bệnh tim cao hơn so với người trẻ tuổi.
+ Trong số những người được khảo sát trong dataset, số người cao tuổi lại khá nhiều so với người trẻ tuổi nhưng số lượng người bị bệnh tim lại cáo hơn nhiều.

In [ ]:
plt.figure(figsize=(10,10))
sns.countplot(data=data, x='HeartDisease', hue='Smoking')

Tuy có sự chênh lệch khá nhiều nhưng:
+ Ta có thể thấy những người không hút thuốc không bị bệnh tim có số lượng khá nhiều.
+ Trong khi những người bị bệnh tim thì số lượng người hút thuốc lại cao hơn một chút so với những người không hút thuốc

In [ ]:
plt.figure(figsize=(10,10))
sns.countplot(data=data, x='HeartDisease', hue='GenHealth')

Những người có kết quả sức khỏe tổng quát từ tốt trở lên đều có tỉ lệ lớn không bị bệnh tim

In [ ]:
plt.figure(figsize=(10,10))
sns.countplot(data=data, x='HeartDisease', hue='GenHealth')